# training/train.py — Training Loop for MA1DCNN (Phase 5, Real Data)

The first place `ma1dcnn.ipynb`'s architecture and `windowing.ipynb`'s
real windowed data actually meet. Builds the dataset/loader, the
training loop, the validation metrics, and the top-level training
function `train_design_questions.docx` and
`pipeline_sequence_and_dependencies.docx` already committed to --
turning that planning into running, tested code.

## Judgment calls this notebook resolves (already answered, not re-opened here)

`train_design_questions.docx` worked through four open questions by
reading Akindoju (2025) directly. This notebook applies those answers
rather than re-deriving them:

1. **Loss function** -- `BCEWithLogitsLoss`. Table 4.4's `Use Focal`
   column is `FALSE` in both the before- and after-tuning best trials --
   plain BCE won empirically, despite focal loss being the headline
   strategy in the praxis's narrative text.
2. **LR scheduler** -- `CosineAnnealingWarmRestarts`, stated explicitly
   in Akindoju's own abstract.
3. **Classification threshold** -- kept as a tunable config value, not
   hardcoded at 0.5. Akindoju's own tuned value was 0.76957 (Table 4.4).
4. **Combined vs. per-file training** -- one combined dataset across all
   9 files, not 9 separate per-file models. The 5-column label vector
   is already what distinguishes fault components; nothing about
   per-file training was found anywhere in Akindoju (2025).

The one question `train_design_questions.docx` left genuinely open --
whether `W` (flight-condition channels) should be included, 28 vs. 32
channels -- is `windowing.ipynb`'s call, not this notebook's:
`INCLUDE_FLIGHT_CONDITIONS=False` there already fixes this notebook's
input at 28 channels, matching `DEFAULT_N_INPUT_CHANNELS` below.

## Build scope for this notebook (agreed decision)

Same discipline as every notebook before it: every piece is built and
fully tested against small, synthetic, in-memory data first -- no
Drive, no real N-CMAPSS data needed to run or understand any cell
below except the final "Run it" section. Per
`pipeline_sequence_and_dependencies.docx`, Section 7, the dataset
loading, training-step, and evaluation logic are each their own
reusable function (`build_dataloaders`, `train_one_epoch`, `evaluate`),
not inlined into one monolithic script -- so `compression/`'s later
quantization-aware fine-tuning and pruning fine-tuning can import and
reuse them instead of duplicating this loop from scratch.

## What every section below does

1. **Constants (shared)** -- repeated verbatim from `ma1dcnn.ipynb` and
   `windowing.ipynb`, standalone convention.
2. **Model** -- `MA1DCNN` / `build_ma1dcnn`, also repeated verbatim,
   already fully proven in `ma1dcnn.ipynb` (not re-tested here).
3. **Constants (new)** -- this notebook's own training-level judgment
   calls: the training hyperparameters from Table 4.4, and the LR
   scheduler's `T_0`/`T_mult`, which Akindoju never states a value for.
4. **`WindowedFaultDataset`** -- loads and concatenates all 9 files'
   windows for one split.
5. **`build_dataloaders`** -- wraps it in train/val `DataLoader`s.
6. **`compute_metrics`** -- F1, precision, recall, Hamming loss, Jaccard
   -- the five metrics `pipeline_sequence_and_dependencies.docx` already
   committed to.
7. **`train_one_epoch`** / **`evaluate`** -- one training pass, one
   validation pass.
8. **`run_training`** -- ties it all together: builds the loaders, the
   model, the optimizer and scheduler, loops over epochs, and applies
   early stopping on validation macro-F1.
9. **Run it** -- the real run against `data/windowed/*.npz`.


## 0. Imports

`torch` and `torch.nn` build and train the model; `numpy` reads the
saved `.npz` windows; `sklearn.metrics` computes the five validation
metrics rather than hand-rolling multi-label F1/Jaccard math; `os` and
`tempfile` are only used for file paths and this notebook's own
synthetic self-checks.

In [1]:
import os
import tempfile

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss, jaccard_score  # the 5 committed metrics

## 1. Constants shared with `ma1dcnn.ipynb` and `windowing.ipynb`

Repeated here verbatim (not imported) -- the same standalone
convention every notebook in this project uses, so this one can run
and be understood with no other file open. These values MUST match
their source notebooks exactly, or the saved `.npz` files and the
model built below would silently disagree about shape or column order.

In [2]:
# Repeated from loaders.ipynb / ma1dcnn.ipynb / windowing.ipynb -- column order fixes which
# label column a fault flag lands in, and its length (5) fixes the model's output width.
FAULT_COLUMNS = ["fan_fail", "lpc_fail", "hpc_fail", "hpt_fail", "lpt_fail"]
N_FAULT_COLUMNS = len(FAULT_COLUMNS)  # 5

# Repeated from loaders.ipynb / windowing.ipynb -- same 9 files, same fixed order.
NCMAPSS_FILES = [
    "N-CMAPSS_DS01-005.h5", "N-CMAPSS_DS02-006.h5", "N-CMAPSS_DS03-012.h5",
    "N-CMAPSS_DS04.h5", "N-CMAPSS_DS05.h5", "N-CMAPSS_DS06.h5",
    "N-CMAPSS_DS07.h5", "N-CMAPSS_DS08a-009.h5", "N-CMAPSS_DS08c-008.h5",
]

# windowing.ipynb's own OUTPUT directory -- this notebook's INPUT. Saved files are named
# "<file>_<split>_windowed.npz" there, which _windowed_file_path() below reproduces exactly.
DEFAULT_WINDOWED_DIR = "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/windowed"

# Repeated from ma1dcnn.ipynb -- needed to construct MA1DCNN below.
CAM_REDUCTION_RATIO = 4

# Repeated from ma1dcnn.ipynb. Fixed at 28 because windowing.ipynb's own
# INCLUDE_FLIGHT_CONDITIONS=False default was used to produce the real data/windowed/ files --
# see train_design_questions.docx for why the 28-vs-32 question stays genuinely open elsewhere,
# and note this is windowing.ipynb's call, not something this notebook decides.
DEFAULT_N_INPUT_CHANNELS = 28

# Table 4.4 (Akindoju, 2025) -- architecture hyperparameters, repeated verbatim from ma1dcnn.ipynb.
BEFORE_TUNING_CONFIG = {
    "n_input_channels": DEFAULT_N_INPUT_CHANNELS,
    "channel_1": 32, "channel_2": 64, "channel_3": 128,
    "kernel_size": 7, "fc_units": 64, "dropout": 0.5,
    "n_outputs": N_FAULT_COLUMNS,
}
AFTER_TUNING_CONFIG = {
    "n_input_channels": DEFAULT_N_INPUT_CHANNELS,
    "channel_1": 32, "channel_2": 128, "channel_3": 128,
    "kernel_size": 5, "fc_units": 16, "dropout": 0.39919,
    "n_outputs": N_FAULT_COLUMNS,
}

## 2. Model -- repeated verbatim from `ma1dcnn.ipynb`

`Conv1dSamePadding`, `ChannelAttentionModule` (CAM),
`ElementwiseAttentionModule` (EAM), `MA1DCNN`, and `build_ma1dcnn` are
copied here unchanged -- same standalone convention as the constants
above. **Deliberately not re-tested in this notebook**: `ma1dcnn.ipynb`
Sections 6-8 already proved the forward-pass shapes, the probability
range, the exact parameter counts against hand-derived formulas, and
the receptive field, empirically. Re-running those same proofs here
would only duplicate that notebook's own test suite, not add new
information -- this notebook's own tests start at Section 4
(`WindowedFaultDataset`), which is what's actually new.

In [3]:
class Conv1dSamePadding(nn.Module):
    """1D convolution that keeps the sequence length unchanged (stride=1, odd kernel)."""

    def __init__(self, in_channels, out_channels, kernel_size, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()
        if kernel_size % 2 == 0:
            raise ValueError(f"Conv1dSamePadding needs an odd kernel_size, got {kernel_size}")
        padding = (kernel_size - 1) // 2  # same-padding formula: P = (K - 1) / 2
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, stride=1, padding=padding)
        # NEW -- synced from ma1dcnn.ipynb: identity (zero params/buffers) when disabled (the
        # default); placed BEFORE activation -- normalizes the raw conv output, not an already-ReLU-clipped one.
        self.bn = nn.BatchNorm1d(out_channels) if use_batchnorm else nn.Identity()
        self.activation = nn.ReLU()
        self.conv_dropout = nn.Dropout(p=conv_dropout)  # p=0.0 default is a no-op, existing configs unaffected

    def forward(self, x):
        return self.conv_dropout(self.activation(self.bn(self.conv(x))))  # NEW -- self.bn is a no-op when use_batchnorm=False


class ChannelAttentionModule(nn.Module):
    """CAM: one learned attention weight per CHANNEL, shared across all timesteps."""

    def __init__(self, n_channels, reduction_ratio=CAM_REDUCTION_RATIO):
        super().__init__()
        hidden = max(1, n_channels // reduction_ratio)
        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excite = nn.Sequential(
            nn.Linear(n_channels, hidden), nn.ReLU(),
            nn.Linear(hidden, n_channels), nn.Sigmoid(),
        )

    def forward(self, x):
        batch, channels, length = x.shape
        squeezed = self.squeeze(x).view(batch, channels)
        channel_weights = self.excite(squeezed).view(batch, channels, 1)
        return x * channel_weights


class ElementwiseAttentionModule(nn.Module):
    """EAM: one learned attention weight per TIMESTEP, shared across all channels."""

    def __init__(self, n_channels):
        super().__init__()
        self.score = nn.Conv1d(in_channels=n_channels, out_channels=1, kernel_size=1)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        attention = self.activation(self.score(x))
        return x * attention


class MA1DCNN(nn.Module):
    """3 conv blocks -> CAM -> EAM -> global average pooling -> FC head -> 5 logits."""

    def __init__(self, n_input_channels, channel_1, channel_2, channel_3,
                 kernel_size, fc_units, dropout, n_outputs,
                 cam_reduction_ratio=CAM_REDUCTION_RATIO, conv_dropout=0.0, use_batchnorm=False):
        super().__init__()
        # conv_dropout/use_batchnorm default to off -- BEFORE_TUNING_CONFIG/AFTER_TUNING_CONFIG/
        # BEFORE_TUNING_CONFIG_CONV_DROPOUT build unaffected models. The two are independent,
        # composable levers -- use_batchnorm=True layers ON TOP of conv_dropout, never replaces it.
        self.conv1 = Conv1dSamePadding(n_input_channels, channel_1, kernel_size, conv_dropout, use_batchnorm)
        self.conv2 = Conv1dSamePadding(channel_1, channel_2, kernel_size, conv_dropout, use_batchnorm)
        self.conv3 = Conv1dSamePadding(channel_2, channel_3, kernel_size, conv_dropout, use_batchnorm)
        self.cam = ChannelAttentionModule(channel_3, reduction_ratio=cam_reduction_ratio)
        self.eam = ElementwiseAttentionModule(channel_3)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(channel_3, fc_units)
        self.fc_activation = nn.ReLU()
        self.output_layer = nn.Linear(fc_units, n_outputs)

    def forward(self, x):
        # x: (batch, n_input_channels, L) -- a batch of windowed, normalized sensor sequences.
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.cam(x)
        x = self.eam(x)
        x = self.global_avg_pool(x)
        x = x.view(x.shape[0], -1)
        x = self.dropout(x)
        x = self.fc(x)
        x = self.fc_activation(x)
        return self.output_layer(x)  # RAW logits -- BCEWithLogitsLoss expects these, not sigmoid(logits)

    def predict_proba(self, x):
        """sigmoid(forward(x)) -- for inference/thresholding, never inside a training loop's backward pass."""
        with torch.no_grad():
            return torch.sigmoid(self.forward(x))


def build_ma1dcnn(config):
    """Construct an MA1DCNN from a plain hyperparameter dict (e.g. BEFORE_TUNING_CONFIG)."""
    return MA1DCNN(
        n_input_channels=config["n_input_channels"], channel_1=config["channel_1"],
        channel_2=config["channel_2"], channel_3=config["channel_3"],
        kernel_size=config["kernel_size"], fc_units=config["fc_units"],
        dropout=config["dropout"], n_outputs=config["n_outputs"],
        conv_dropout=config.get("conv_dropout", 0.0),      # .get() -- NOT config[...] -- for backward compatibility
        use_batchnorm=config.get("use_batchnorm", False),  # same -- NEW, synced from ma1dcnn.ipynb
    )

## 3. Constants this notebook introduces -- the training-level judgment calls

`BEFORE_TUNING_CONFIG` / `AFTER_TUNING_CONFIG` above only cover
*architecture* hyperparameters. Table 4.4 also reports *training*
hyperparameters -- learning rate, weight decay, batch size, and the
classification threshold -- which belong here, not in `ma1dcnn.ipynb`.
Named explicitly, the same discipline `windowing.ipynb` used for its
four judgment calls.

In [4]:
# Table 4.4 (Akindoju, 2025) -- TRAINING hyperparameters (separate from the architecture
# configs above). "Use Focal" is kept here for traceability against the table, even though
# Section 9's run_training() below never branches on it -- see the markdown note just before
# run_training for why.
TRAIN_BEFORE_TUNING_CONFIG = {
    "learning_rate": 1e-3,       # Table 4.4: Learning Rate (before tuning)
    "weight_decay": 1e-4,        # Table 4.4: Weight Decay (before tuning)
    "batch_size": 128,           # Table 4.4: Batch Size (before tuning)
    "threshold": 0.5,            # Table 4.4: Threshold (before tuning) -- the conventional default
    "use_focal": False,          # Table 4.4: Use Focal (before tuning)
}
TRAIN_AFTER_TUNING_CONFIG = {
    "learning_rate": 0.00019963,  # Table 4.4: Learning Rate (after tuning)
    "weight_decay": 4.29e-06,     # Table 4.4: Weight Decay (after tuning)
    "batch_size": 256,            # Table 4.4: Batch Size (after tuning)
    "threshold": 0.76957,         # Table 4.4: Threshold (after tuning) -- Optuna's tuned value
    "use_focal": False,           # Table 4.4: Use Focal (after tuning) -- unchanged by tuning
}

# New judgment call (not from Table 4.4): same architecture and same base training
# settings as TRAIN_BEFORE_TUNING_CONFIG, but with weight_decay raised 10x. Added after
# the real Oct 2026 run (Section 10) showed val_loss rising from epoch 1 onward while
# train_loss kept falling -- immediate overfitting -- to test whether stronger L2 alone
# fixes it, with the architecture held fixed (BEFORE_TUNING_CONFIG) so this is a clean,
# single-variable experiment rather than being tangled up with AFTER_TUNING_CONFIG's
# simultaneous architecture changes.
TRAIN_REGULARIZED_CONFIG = {
    "learning_rate": 1e-3,
    "weight_decay": 1e-3,   # 10x TRAIN_BEFORE_TUNING_CONFIG's 1e-4
    "batch_size": 128,
    "threshold": 0.5,
    "use_focal": False,
}

# Synced from ma1dcnn.ipynb -- same architecture as BEFORE_TUNING_CONFIG, but with dropout
# added inside the conv stack (0.2 on each of the 3 conv blocks), not just once before the
# FC layer. Added after train.ipynb's own weight_decay experiments (Section 10): neither
# coupled Adam nor decoupled AdamW weight_decay alone closed the generalization gap -- see
# overfitting_results_and_options.docx and the AdamW verdict (worse on nearly every metric,
# gap more than 3x the regularized run's). conv_dropout defaults to 0.0 everywhere else, so
# this is strictly additive -- BEFORE_TUNING_CONFIG/AFTER_TUNING_CONFIG/TRAIN_REGULARIZED_CONFIG
# are all untouched.
BEFORE_TUNING_CONFIG_CONV_DROPOUT = dict(BEFORE_TUNING_CONFIG, conv_dropout=0.2)

# Synced from ma1dcnn.ipynb -- same architecture as BEFORE_TUNING_CONFIG_CONV_DROPOUT, but
# with BatchNorm1d added after each conv block (LAYERED ON TOP of conv_dropout=0.2, not
# replacing it). Added after ma1dcnn.ipynb's own synthetic train/eval running-statistics
# test passed -- see that notebook's Section 2 for the isolated-layer test and the
# integration test after its Section 6. use_batchnorm defaults to False everywhere else, so
# this is strictly additive -- every config above is untouched.
BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN = dict(BEFORE_TUNING_CONFIG_CONV_DROPOUT, use_batchnorm=True)

# New judgment call: CosineAnnealingWarmRestarts is confirmed by Akindoju's own abstract, but
# its T_0 (epochs until the first restart) and T_mult (how much each restart's period grows by)
# are NOT stated anywhere in the praxis -- these are this notebook's own reasonable defaults,
# named here so they can be revisited without hunting through run_training()'s body.
SCHEDULER_T_0 = 10
SCHEDULER_T_MULT = 2

# Training-loop bookkeeping -- also not specified anywhere in Akindoju (2025).
N_EPOCHS = 30                 # upper bound; early stopping usually ends a run sooner
EARLY_STOPPING_PATIENCE = 5   # epochs allowed with no val_f1_macro improvement before stopping
RANDOM_SEED = 0                # for reproducible weight init and training order

# Picks the GPU automatically when Colab provides one, falls back to CPU otherwise --
# never hardcoded, so this notebook runs unmodified on either.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Trial 13 from training/optuna_search.ipynb's completed 30-trial multi-objective search
# (directions=["maximize", "minimize"] over val_f1_macro / max generalization gap, TPE sampler,
# N_SEEDS_PER_TRIAL=2, real N-CMAPSS windowed data). study.best_trials / print_pareto_front(study)
# returned a 5-trial Pareto front (trials 9, 1, 12, 13, 4); trial 13 is the recommended pick --
# a knee-point calculation (Branke, Deb, Dierolf, & Osswald, 2004) over the front's marginal
# F1-per-gap exchange rate shows a sharp drop right after it. Both trial 13 (val_f1_macro=0.8375,
# max_gap=-0.0188) and trial 12 (0.8445, -0.0089) Pareto-dominate every hand-run config above
# outright -- see overfitting_adamw_comparison.xlsx's 'Optuna Search Results' sheet for the full
# front and the exchange-rate math. channel_1=32 and channel_3=128 were held fixed by the search
# (not part of its space), so both are inherited from BEFORE_TUNING_CONFIG unchanged, same as
# every Optuna trial. use_focal was never part of the search space either -- left False, matching
# every config above.
TRIAL_13_MODEL_CONFIG = dict(
    BEFORE_TUNING_CONFIG,
    channel_2=64, kernel_size=5, fc_units=16,
    dropout=0.5945728900118123,
    conv_dropout=0.33869914326562556,
    use_batchnorm=False,
)
TRIAL_13_TRAIN_CONFIG = {
    "learning_rate": 0.0004689002238683733,
    "weight_decay": 0.002164379191774169,
    "batch_size": 256,
    "threshold": 0.4214610774407138,
    "use_focal": False,
}


## 4. `WindowedFaultDataset` -- load and concatenate all 9 files' windows

Judgment call 4 from `train_design_questions.docx`: ONE combined
dataset across all 9 files, not 9 separate per-file datasets. Each file
contributes windows from a different fault component mix, and the
5-column label vector is already what tells them apart -- there's
nothing in Akindoju (2025) suggesting per-file models.

In [5]:
def _windowed_file_path(filename, windowed_dir, split):
    """Reproduce windowing.ipynb's own save-path convention exactly: '<file>_<split>_windowed.npz'."""
    out_name = filename.replace(".h5", f"_{split}_windowed.npz")
    return os.path.join(windowed_dir, out_name)


class WindowedFaultDataset(torch.utils.data.Dataset):
    """All 9 files' windows for ONE split (train or val), concatenated into a single dataset.

    Loaded fully into memory at construction time -- windowing.ipynb's real output is roughly
    710k train windows x 28 channels x 50 timesteps, a few GB as float32, well within a normal
    Colab runtime's RAM. If this ever stops being true, this is the class to make lazy/chunked.
    """

    def __init__(self, windowed_dir=DEFAULT_WINDOWED_DIR, split="train", files=None):
        files = files if files is not None else NCMAPSS_FILES
        X_parts, y_parts, unit_id_parts, start_cycle_parts = [], [], [], []

        for filename in files:
            path = _windowed_file_path(filename, windowed_dir, split)
            with np.load(path) as npz:
                X_parts.append(npz["X"])              # (n_windows_i, 28, 50)
                y_parts.append(npz["y"])              # (n_windows_i, 5)
                unit_id_parts.append(npz["unit_id"])  # (n_windows_i,) -- traceability only
                start_cycle_parts.append(npz["start_cycle"])

        # float32 for X (standard for model inputs) and for y too -- BCEWithLogitsLoss needs a
        # FLOAT target, not the int8 labels windowing.ipynb saved; casting once here, at load
        # time, means every batch downstream is already the right dtype.
        self.X = np.concatenate(X_parts, axis=0).astype(np.float32)
        self.y = np.concatenate(y_parts, axis=0).astype(np.float32)
        self.unit_id = np.concatenate(unit_id_parts, axis=0)
        self.start_cycle = np.concatenate(start_cycle_parts, axis=0)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        # torch.from_numpy shares memory with self.X/self.y rather than copying -- fine here
        # since DataLoader workers only ever read, never write, these tensors.
        return torch.from_numpy(self.X[idx]), torch.from_numpy(self.y[idx])

In [6]:
def _test_windowed_fault_dataset_concatenates_all_files():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5", "FAKE_B.h5"]
        sizes = {"FAKE_A.h5": 7, "FAKE_B.h5": 3}
        rng = np.random.default_rng(0)

        for filename, n in sizes.items():
            X = rng.random((n, 28, 50)).astype(np.float32)
            y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)  # saved as int8, like windowing.ipynb does
            unit_id = np.arange(n)
            start_cycle = np.arange(n) * 50
            np.savez_compressed(_windowed_file_path(filename, windowed_dir, "train"),
                                 X=X, y=y, unit_id=unit_id, start_cycle=start_cycle)

        ds = WindowedFaultDataset(windowed_dir=windowed_dir, split="train", files=files)
        assert len(ds) == sum(sizes.values()), len(ds)  # 7 + 3 = 10, both files present
        assert ds.X.dtype == np.float32
        assert ds.y.dtype == np.float32  # cast from int8 -- confirms the dtype fix for BCEWithLogitsLoss

        x0, y0 = ds[0]
        assert x0.shape == (28, 50), x0.shape
        assert y0.shape == (5,), y0.shape
        assert x0.dtype == torch.float32
        assert y0.dtype == torch.float32

        # FAKE_B.h5's rows must appear at the tail, in file order -- confirms concatenation order.
        x_last, _ = ds[len(ds) - 1]
        assert np.allclose(x_last.numpy(), ds.X[-1])

    print("WindowedFaultDataset: PASSED (concatenates all files, correct dtypes and shapes)")


_test_windowed_fault_dataset_concatenates_all_files()

WindowedFaultDataset: PASSED (concatenates all files, correct dtypes and shapes)


## 5. `build_dataloaders` -- the reusable loader-building function

Train shuffles every epoch (so batches aren't always in the same
order); validation never shuffles (so validation metrics are
reproducible run to run). Per
`pipeline_sequence_and_dependencies.docx` Section 7, this is also the
function `compression/`'s post-training quantization step will later
reuse to get a calibration batch -- another reason it stands alone
rather than being inlined into `run_training` below.

In [7]:
def build_dataloaders(windowed_dir=DEFAULT_WINDOWED_DIR, batch_size=TRAIN_BEFORE_TUNING_CONFIG["batch_size"],
                       files=None, num_workers=0):
    """Build train and val DataLoaders over WindowedFaultDataset.

    num_workers=0 by default -- safe everywhere, including Colab. Raise it once this is
    confirmed to run end to end if data loading becomes the training bottleneck.
    """
    train_dataset = WindowedFaultDataset(windowed_dir=windowed_dir, split="train", files=files)
    val_dataset = WindowedFaultDataset(windowed_dir=windowed_dir, split="val", files=files)

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)   # shuffled every epoch
    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)    # never shuffled
    return train_loader, val_loader

In [8]:
def _test_build_dataloaders_shuffles_train_not_val():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5"]
        rng = np.random.default_rng(1)
        n = 20
        for split in ["train", "val"]:
            X = rng.random((n, 28, 50)).astype(np.float32)
            y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)
            np.savez_compressed(_windowed_file_path("FAKE_A.h5", windowed_dir, split),
                                 X=X, y=y, unit_id=np.arange(n), start_cycle=np.arange(n))

        train_loader, val_loader = build_dataloaders(windowed_dir=windowed_dir, batch_size=4, files=files)

        # The sampler TYPE is the real contract here (RandomSampler vs SequentialSampler) --
        # checking it directly is more reliable than checking batch order, which could coincide.
        assert isinstance(train_loader.sampler, torch.utils.data.RandomSampler), type(train_loader.sampler)
        assert isinstance(val_loader.sampler, torch.utils.data.SequentialSampler), type(val_loader.sampler)
        assert len(train_loader.dataset) == n
        assert len(val_loader.dataset) == n

    print("build_dataloaders: PASSED (train uses RandomSampler, val uses SequentialSampler)")


_test_build_dataloaders_shuffles_train_not_val()

build_dataloaders: PASSED (train uses RandomSampler, val uses SequentialSampler)


## 6. `compute_metrics` -- the five validation metrics

`pipeline_sequence_and_dependencies.docx` Section 6 already committed
to F1, precision, recall, Hamming loss, and Jaccard -- all standard
multi-label metrics, computed by thresholding `sigmoid(logits)` before
comparing to the 5-column label vector. `zero_division=0` matters here:
with 5 independent fault columns and small batches, it's common for a
batch to have zero true positives in some column, which would otherwise
raise a warning (or crash, depending on sklearn version) instead of
just scoring that column 0.

In [9]:
def compute_metrics(y_true, logits, threshold):
    """Threshold sigmoid(logits) at `threshold` and score against y_true.

    Parameters
    ----------
    y_true : torch.Tensor, shape (n, 5)
    logits : torch.Tensor, shape (n, 5) -- RAW logits, same convention as MA1DCNN.forward()
    threshold : float -- e.g. TRAIN_BEFORE_TUNING_CONFIG["threshold"] (0.5) or the tuned 0.76957
    """
    probs = torch.sigmoid(logits)                            # (n, 5), each entry independently in [0, 1]
    y_pred = (probs >= threshold).to(torch.int64).cpu().numpy()
    y_true_np = y_true.to(torch.int64).cpu().numpy()

    return {
        "f1_macro": f1_score(y_true_np, y_pred, average="macro", zero_division=0),
        "precision_macro": precision_score(y_true_np, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true_np, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true_np, y_pred),      # fraction of (sample, label) entries that disagree
        "jaccard_macro": jaccard_score(y_true_np, y_pred, average="macro", zero_division=0),
    }

In [10]:
def _test_compute_metrics_known_cases():
    # Every column has at least one positive AND one negative row -- otherwise sklearn's
    # zero_division=0 convention scores an all-negative column's F1 as 0 even when the
    # prediction for it is perfect, which would make this "perfect prediction" case fail
    # for a reason that has nothing to do with compute_metrics() being wrong.
    y_true = torch.tensor([
        [1, 0, 1, 0, 1],
        [0, 1, 0, 1, 0],
        [1, 1, 0, 0, 1],
        [0, 0, 1, 1, 0],
    ], dtype=torch.float32)

    # Perfect prediction: huge positive logits exactly where y_true==1, huge negative elsewhere
    # -- sigmoid(10) and sigmoid(-10) are indistinguishable from 1 and 0 at any sane threshold.
    perfect_logits = torch.where(y_true == 1, torch.tensor(10.0), torch.tensor(-10.0))
    m = compute_metrics(y_true, perfect_logits, threshold=0.5)
    assert m["f1_macro"] == 1.0, m
    assert m["precision_macro"] == 1.0, m
    assert m["recall_macro"] == 1.0, m
    assert m["hamming_loss"] == 0.0, m
    assert m["jaccard_macro"] == 1.0, m

    # Completely inverted prediction: confidently wrong everywhere.
    wrong_logits = torch.where(y_true == 1, torch.tensor(-10.0), torch.tensor(10.0))
    m_wrong = compute_metrics(y_true, wrong_logits, threshold=0.5)
    assert m_wrong["f1_macro"] == 0.0, m_wrong
    assert m_wrong["hamming_loss"] == 1.0, m_wrong
    assert m_wrong["jaccard_macro"] == 0.0, m_wrong

    print("compute_metrics: PASSED (perfect prediction -> 1.0/0.0, fully inverted prediction -> 0.0/1.0)")


_test_compute_metrics_known_cases()

compute_metrics: PASSED (perfect prediction -> 1.0/0.0, fully inverted prediction -> 0.0/1.0)


## 7. `train_one_epoch` -- one pass over the training set

Standard PyTorch loop: zero gradients, forward pass, compute loss,
backward pass, optimizer step. Returns the average loss across all
batches, purely for logging -- `run_training` below is what actually
decides whether training is working, using validation metrics.

In [11]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    """One full pass over `loader`, updating `model`'s weights. Returns the average training loss."""
    model.train()  # enables dropout -- the opposite of model.eval() used everywhere in ma1dcnn.ipynb's tests
    total_loss, n_batches = 0.0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()          # PyTorch accumulates gradients by default -- must clear them each step
        logits = model(X)              # (batch, 5) raw logits
        loss = loss_fn(logits, y)      # BCEWithLogitsLoss -- expects raw logits, applies sigmoid internally
        loss.backward()                # computes gradients for every parameter via autograd
        optimizer.step()                # applies one Adam update using those gradients

        total_loss += loss.item()      # .item() pulls a plain Python float out of the 0-d loss tensor
        n_batches += 1

    return total_loss / n_batches

In [12]:
def _test_train_one_epoch_reduces_loss_on_a_tiny_batch():
    # The standard "can this training step learn anything at all" sanity check: repeatedly
    # train on the SAME small, fixed batch and confirm the loss falls. If it didn't, something
    # upstream (gradients not flowing, optimizer misconfigured, loss/model mismatch) is broken --
    # this test would catch that long before any real, slow training run would.
    torch.manual_seed(0)
    config = dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8, dropout=0.0)
    model = build_ma1dcnn(config).to(DEVICE)

    X = torch.randn(16, 28, 50)
    y = (torch.rand(16, 5) > 0.5).float()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=4)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    loss_fn = nn.BCEWithLogitsLoss()

    first_loss = train_one_epoch(model, loader, optimizer, loss_fn, DEVICE)
    for _ in range(19):
        last_loss = train_one_epoch(model, loader, optimizer, loss_fn, DEVICE)

    assert last_loss < first_loss, (first_loss, last_loss)
    assert last_loss == last_loss  # NaN != NaN -- this also fails loudly if training diverged

    print(f"train_one_epoch: PASSED (loss fell from {first_loss:.4f} to {last_loss:.4f} over 20 epochs on a fixed tiny batch)")


_test_train_one_epoch_reduces_loss_on_a_tiny_batch()

train_one_epoch: PASSED (loss fell from 0.6930 to 0.5627 over 20 epochs on a fixed tiny batch)


## 8. `evaluate` -- one pass over the validation set

Collects every batch's logits and labels FIRST, then calls
`compute_metrics` exactly once on the full concatenated set --
deliberately not averaging per-batch metrics, since F1 and Jaccard
don't average correctly across batches (a batch with zero true
positives would distort a naive per-batch average in a way that
disappears once all predictions are pooled first).

In [13]:
def evaluate(model, loader, loss_fn, device, threshold):
    """One full pass over `loader` with gradients disabled. Returns a dict of metrics plus 'loss'."""
    model.eval()  # disables dropout -- deterministic forward passes for a fair evaluation
    total_loss, n_batches = 0.0, 0
    all_logits, all_y = [], []

    with torch.no_grad():  # no autograd bookkeeping needed -- saves memory and time during evaluation
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)
            total_loss += loss.item()
            n_batches += 1
            all_logits.append(logits.cpu())  # move off GPU immediately -- keeps GPU memory free during accumulation
            all_y.append(y.cpu())

    all_logits = torch.cat(all_logits, dim=0)  # every validation window's logits, in one tensor
    all_y = torch.cat(all_y, dim=0)
    metrics = compute_metrics(all_y, all_logits, threshold)  # computed ONCE, over the whole split -- see markdown above
    metrics["loss"] = total_loss / n_batches
    return metrics

In [14]:
def _test_evaluate_returns_expected_keys_and_ranges():
    torch.manual_seed(0)
    model = build_ma1dcnn(dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8)).to(DEVICE)
    X = torch.randn(13, 28, 50)  # 13 -- deliberately not a multiple of batch_size, to exercise a ragged last batch
    y = (torch.rand(13, 5) > 0.5).float()
    loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=4)

    metrics = evaluate(model, loader, nn.BCEWithLogitsLoss(), DEVICE, threshold=0.5)

    expected_keys = {"f1_macro", "precision_macro", "recall_macro", "hamming_loss", "jaccard_macro", "loss"}
    assert set(metrics.keys()) == expected_keys, metrics.keys()
    for key in ["f1_macro", "precision_macro", "recall_macro", "hamming_loss", "jaccard_macro"]:
        assert 0.0 <= metrics[key] <= 1.0, (key, metrics[key])
    assert metrics["loss"] > 0, metrics["loss"]

    print("evaluate: PASSED (expected keys present, all metrics in [0,1], handles a ragged final batch)")


_test_evaluate_returns_expected_keys_and_ranges()

evaluate: PASSED (expected keys present, all metrics in [0,1], handles a ragged final batch)


## 9. `run_training` -- the full loop, tying everything together

Builds the loaders, the model, the optimizer, and the scheduler; loops
over epochs calling `train_one_epoch` then `evaluate`; prints both
every epoch (the same "never silent, one step at a time" discipline
every notebook in this project uses); tracks the best validation
macro-F1 seen so far; and stops early if it hasn't improved for
`patience` epochs. The function **returns the best checkpoint**, not
necessarily the last epoch's weights -- early stopping needs a few bad
epochs to confirm things have stopped improving, so the last epoch run
is often slightly worse than the best one.

**On `use_focal`:** `TRAIN_BEFORE_TUNING_CONFIG`/`TRAIN_AFTER_TUNING_CONFIG`
both carry a `use_focal` key, kept for traceability against Table 4.4 --
but `run_training` below never branches on it and always uses plain
`BCEWithLogitsLoss`. This is deliberate, not an oversight: Akindoju's
own best trial turned focal loss OFF in both the before- and
after-tuning columns (see `train_design_questions.docx`, Section 4), so
implementing a focal-loss branch would add real complexity for a loss
function that didn't win empirically. If a later experiment wants to
try it anyway, this docstring marks the exact spot to add that branch.

In [15]:
def run_training(model_config, train_config, windowed_dir=DEFAULT_WINDOWED_DIR, n_epochs=N_EPOCHS,
                  patience=EARLY_STOPPING_PATIENCE, device=DEVICE, files=None, verbose=True,
                  optimizer_cls=torch.optim.Adam):
    """Build everything, train with early stopping, return the BEST model and the full per-epoch history.

    Parameters
    ----------
    model_config : dict -- e.g. BEFORE_TUNING_CONFIG, passed to build_ma1dcnn()
    train_config : dict -- e.g. TRAIN_BEFORE_TUNING_CONFIG (learning_rate, weight_decay, batch_size, threshold)
    optimizer_cls : type -- torch.optim.Adam (default, matches every run so far) or torch.optim.AdamW.
        Adam applies weight_decay as coupled L2 (added into the gradient before Adam's adaptive
        per-parameter scaling); AdamW decouples it, applying weight_decay directly to the parameter
        instead (Loshchilov & Hutter, 2019). Swapping this in is a single-variable way to test
        whether TRAIN_REGULARIZED_CONFIG's weight_decay=1e-3 regularizes more effectively when
        decoupled from Adam's adaptive scaling -- see overfitting_results_and_options.docx Section 4.2.

    Returns
    -------
    model : MA1DCNN -- weights restored to the BEST epoch's checkpoint (highest val_f1_macro)
    history : list[dict] -- one record per epoch actually run: epoch, train_loss, and val_* for every metric
    """
    torch.manual_seed(RANDOM_SEED)  # reproducible weight init and (with shuffle=True) batch order

    train_loader, val_loader = build_dataloaders(
        windowed_dir=windowed_dir, batch_size=train_config["batch_size"], files=files)

    model = build_ma1dcnn(model_config).to(device)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = optimizer_cls(model.parameters(), lr=train_config["learning_rate"],
                               weight_decay=train_config["weight_decay"])
    # Resolved judgment call (train_design_questions.docx, Section 4): CosineAnnealingWarmRestarts,
    # per Akindoju's own abstract. T_0/T_mult are this notebook's own named defaults (Section 3).
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=SCHEDULER_T_0, T_mult=SCHEDULER_T_MULT)

    history = []
    best_f1 = -1.0           # guaranteed to be beaten by epoch 1's real (>=0) f1_macro
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, n_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        val_metrics = evaluate(model, val_loader, loss_fn, device, threshold=train_config["threshold"])
        scheduler.step()  # advances the cosine schedule -- called once per EPOCH, not per batch

        record = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
        history.append(record)

        if verbose:
            print(f"epoch {epoch:>3}/{n_epochs} | train_loss {train_loss:.4f} | val_loss {val_metrics['loss']:.4f} "
                  f"| val_f1_macro {val_metrics['f1_macro']:.4f} | val_hamming {val_metrics['hamming_loss']:.4f}")

        if val_metrics["f1_macro"] > best_f1:  # strictly greater -- a TIE does not count as improvement
            best_f1 = val_metrics["f1_macro"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}  # clone -- a snapshot, not a live reference
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                if verbose:
                    print(f"early stopping: no val_f1_macro improvement for {patience} epochs")
                break

    model.load_state_dict(best_state)  # return the BEST checkpoint, not whatever epoch the loop happened to end on
    return model, history

### Synthetic data for the two self-checks below

Both tests need realistic-shaped `data/windowed/*.npz` files on disk,
generated rather than copied from real data -- `learnable=True` builds
a task the model can actually solve (each fault column is controlled
by a clear mean shift in one input channel, plus noise); `learnable=False`
builds a label with no relationship to `X` at all, used only to force
the early-stopping branch deterministically below.

In [16]:
def _make_synthetic_windowed_files(windowed_dir, files, n_train, n_val, learnable=True, seed=0):
    rng = np.random.default_rng(seed)
    for filename in files:
        for split, n in [("train", n_train), ("val", n_val)]:
            if learnable:
                # Generative, not discriminative: draw y FIRST, then build channel j (j=0..4) with
                # a strong mean shift (+2.0 if y[:,j]==1 else -2.0) plus unit noise -- a clear,
                # reliably learnable signal for global-average-pooling + FC to pick up.
                y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)
                X = rng.standard_normal((n, 28, 50)).astype(np.float32)
                for j in range(5):
                    shift = np.where(y[:, j] == 1, 2.0, -2.0).astype(np.float32)
                    X[:, j, :] += shift[:, None]
            else:
                # label has NOTHING to do with X -- unlearnable by construction.
                X = rng.standard_normal((n, 28, 50)).astype(np.float32)
                y = rng.integers(0, 2, size=(n, 5)).astype(np.int8)
            unit_id = np.arange(n)
            start_cycle = np.arange(n) * 50
            np.savez_compressed(_windowed_file_path(filename, windowed_dir, split),
                                 X=X, y=y, unit_id=unit_id, start_cycle=start_cycle)

In [17]:
def _test_run_training_learns_a_separable_synthetic_task():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5", "FAKE_B.h5"]
        _make_synthetic_windowed_files(windowed_dir, files, n_train=200, n_val=60, learnable=True, seed=0)

        small_model_config = dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8, dropout=0.1)
        fast_train_config = dict(TRAIN_BEFORE_TUNING_CONFIG, learning_rate=1e-2, weight_decay=0.0, batch_size=32)

        model, history = run_training(small_model_config, fast_train_config, windowed_dir=windowed_dir,
                                       n_epochs=15, patience=5, files=files, verbose=True)

        assert 1 <= len(history) <= 15
        # run_training restores the BEST checkpoint into `model`, not necessarily the last epoch's
        # (early stopping can keep running a few epochs past the peak before giving up) -- so the
        # right check is the best val_f1_macro seen, confirmed by re-evaluating the ACTUALLY
        # RETURNED model and checking it matches -- proof load_state_dict(best_state) really took effect.
        best_f1_from_history = max(h["val_f1_macro"] for h in history)
        assert best_f1_from_history > 0.7, f"expected run_training to learn the separable synthetic task well, got best val_f1_macro={best_f1_from_history}"

        _, val_loader = build_dataloaders(windowed_dir=windowed_dir, batch_size=32, files=files)
        recheck = evaluate(model, val_loader, nn.BCEWithLogitsLoss(), DEVICE, threshold=fast_train_config["threshold"])
        assert abs(recheck["f1_macro"] - best_f1_from_history) < 1e-6, (recheck["f1_macro"], best_f1_from_history)

    print(f"run_training: PASSED (learned separable synthetic task, best val_f1_macro={best_f1_from_history:.4f}, "
          f"returned model confirmed to be the best checkpoint not just the last epoch)")


_test_run_training_learns_a_separable_synthetic_task()

epoch   1/15 | train_loss 0.6265 | val_loss 0.5697 | val_f1_macro 0.7277 | val_hamming 0.2983
epoch   2/15 | train_loss 0.4951 | val_loss 0.4350 | val_f1_macro 0.7658 | val_hamming 0.2483
epoch   3/15 | train_loss 0.3753 | val_loss 0.3667 | val_f1_macro 0.8100 | val_hamming 0.2033


epoch   4/15 | train_loss 0.3296 | val_loss 0.3084 | val_f1_macro 0.8416 | val_hamming 0.1667
epoch   5/15 | train_loss 0.2824 | val_loss 0.2775 | val_f1_macro 0.8573 | val_hamming 0.1500
epoch   6/15 | train_loss 0.2667 | val_loss 0.2520 | val_f1_macro 0.8768 | val_hamming 0.1317


epoch   7/15 | train_loss 0.2368 | val_loss 0.2372 | val_f1_macro 0.8829 | val_hamming 0.1267
epoch   8/15 | train_loss 0.2244 | val_loss 0.2225 | val_f1_macro 0.8883 | val_hamming 0.1183
epoch   9/15 | train_loss 0.2210 | val_loss 0.2206 | val_f1_macro 0.8878 | val_hamming 0.1183


epoch  10/15 | train_loss 0.2171 | val_loss 0.2193 | val_f1_macro 0.8853 | val_hamming 0.1217
epoch  11/15 | train_loss 0.2227 | val_loss 0.2019 | val_f1_macro 0.9019 | val_hamming 0.1117
epoch  12/15 | train_loss 0.2072 | val_loss 0.1980 | val_f1_macro 0.9197 | val_hamming 0.0950


epoch  13/15 | train_loss 0.1919 | val_loss 0.1686 | val_f1_macro 0.9396 | val_hamming 0.0683
epoch  14/15 | train_loss 0.1785 | val_loss 0.1613 | val_f1_macro 0.9420 | val_hamming 0.0683
epoch  15/15 | train_loss 0.1945 | val_loss 0.1660 | val_f1_macro 0.9384 | val_hamming 0.0750


run_training: PASSED (learned separable synthetic task, best val_f1_macro=0.9420, returned model confirmed to be the best checkpoint not just the last epoch)


In [18]:
def _test_run_training_early_stopping_triggers():
    with tempfile.TemporaryDirectory() as windowed_dir:
        files = ["FAKE_A.h5"]
        _make_synthetic_windowed_files(windowed_dir, files, n_train=40, n_val=20, learnable=True, seed=1)

        small_model_config = dict(BEFORE_TUNING_CONFIG, channel_1=8, channel_2=16, channel_3=16, fc_units=8)
        # learning_rate=0.0 -- weights never move, so val_f1_macro is IDENTICAL every epoch after
        # the first. With strict ">" for "improved" in run_training, epoch 2 onward never counts
        # as an improvement, so patience=2 must stop training at exactly epoch 3 -- a fully
        # deterministic way to exercise the early-stopping branch, with no dependence on whether
        # real learning happens to plateau on any particular epoch.
        frozen_train_config = dict(TRAIN_BEFORE_TUNING_CONFIG, learning_rate=0.0, weight_decay=0.0, batch_size=8)

        model, history = run_training(small_model_config, frozen_train_config, windowed_dir=windowed_dir,
                                       n_epochs=10, patience=2, files=files, verbose=True)

        assert len(history) == 3, f"expected early stopping at exactly epoch 3, got {len(history)} epochs"

    print("run_training: PASSED (early stopping triggers at exactly the configured patience)")


_test_run_training_early_stopping_triggers()

epoch   1/10 | train_loss 0.6978 | val_loss 0.6996 | val_f1_macro 0.3870 | val_hamming 0.4600
epoch   2/10 | train_loss 0.6981 | val_loss 0.6996 | val_f1_macro 0.3870 | val_hamming 0.4600
epoch   3/10 | train_loss 0.6981 | val_loss 0.6996 | val_f1_macro 0.3870 | val_hamming 0.4600
early stopping: no val_f1_macro improvement for 2 epochs
run_training: PASSED (early stopping triggers at exactly the configured patience)


## 10. Run it

Trains the real `BEFORE_TUNING_CONFIG` architecture on the real,
windowed `data/windowed/*.npz` files -- the first time this project's
model and this project's real data actually meet.

**Run for real against actual Drive data (October 2026).** Early
stopping (`patience=5`) ended training after 6 of the allotted 30
epochs: `val_f1_macro` peaked at **epoch 1** and never improved on
that again, while `train_loss` kept falling every epoch. That gap --
training loss still dropping while validation metrics get worse -- is
overfitting, starting almost immediately. `run_training` behaved
exactly as designed: it detected that epoch 1 was never beaten,
stopped instead of wasting 24 more epochs, and the `model` returned
here is the epoch-1 checkpoint, not epoch 6's.

This cell has been re-executed more than once across this project's
real Colab sessions; the exact numbers differ slightly run to run
(CPU multi-threaded BLAS is not bit-reproducible, and this notebook
has also run on different hardware across sessions) even though the
shape of the result -- immediate overfitting, best at epoch 1, early
stop around epoch 6 -- is consistent. See
`overfitting_adamw_comparison.xlsx` for the full run-to-run comparison
across every cell in this section.

This is the **untuned** baseline, so a result this exact is expected
to leave room on the table -- the gap between `BEFORE_TUNING_CONFIG`
and `AFTER_TUNING_CONFIG` in Table 4.4 is largely about reining in
overfitting like this. That comparison, and `training/optuna_search.py`,
are the natural next step (see **What's next** below).


In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
model, history = run_training(
    model_config=BEFORE_TUNING_CONFIG,
    train_config=TRAIN_BEFORE_TUNING_CONFIG,
    windowed_dir=DEFAULT_WINDOWED_DIR,
    n_epochs=N_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
)
history  # one dict per epoch actually run -- train_loss and every val_* metric, for plotting or inspection

epoch   1/30 | train_loss 0.2428 | val_loss 0.1652 | val_f1_macro 0.7995 | val_hamming 0.0681
epoch   2/30 | train_loss 0.1668 | val_loss 0.1936 | val_f1_macro 0.7910 | val_hamming 0.0704
epoch   3/30 | train_loss 0.1477 | val_loss 0.2206 | val_f1_macro 0.7713 | val_hamming 0.0813
epoch   4/30 | train_loss 0.1344 | val_loss 0.2562 | val_f1_macro 0.7599 | val_hamming 0.0876
epoch   5/30 | train_loss 0.1245 | val_loss 0.2589 | val_f1_macro 0.7925 | val_hamming 0.0710
epoch   6/30 | train_loss 0.1148 | val_loss 0.3419 | val_f1_macro 0.7447 | val_hamming 0.0935
early stopping: no val_f1_macro improvement for 5 epochs


[{'epoch': 1,
  'train_loss': 0.24280323845716414,
  'val_f1_macro': 0.7994548800607877,
  'val_precision_macro': 0.8027477527899662,
  'val_recall_macro': 0.8140828646678357,
  'val_hamming_loss': 0.06812538175673748,
  'val_jaccard_macro': np.float64(0.6827523378619615),
  'val_loss': 0.1652037681022198},
 {'epoch': 2,
  'train_loss': 0.16681205790567802,
  'val_f1_macro': 0.7909954941618161,
  'val_precision_macro': 0.8064932959807496,
  'val_recall_macro': 0.7941553470831364,
  'val_hamming_loss': 0.07041106994639708,
  'val_jaccard_macro': np.float64(0.6759246609235954),
  'val_loss': 0.19359050512866155},
 {'epoch': 3,
  'train_loss': 0.14771410121455542,
  'val_f1_macro': 0.7713347502650705,
  'val_precision_macro': 0.74492296325285,
  'val_recall_macro': 0.8116689733844208,
  'val_hamming_loss': 0.08132460311570583,
  'val_jaccard_macro': np.float64(0.6423402868500256),
  'val_loss': 0.2205780516659218},
 {'epoch': 4,
  'train_loss': 0.1344067974602931,
  'val_f1_macro': 0.7598

In [28]:
torch.save(model.state_dict(), "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/ma1dcnn_before_tuning.pt")
print("saved trained model weights")

saved trained model weights


### Optional: same architecture, stronger regularization

`BEFORE_TUNING_CONFIG` (architecture, unchanged) + `TRAIN_REGULARIZED_CONFIG`
(Section 3) -- a single-variable experiment next to the run above. Everything
is identical to Section 10's real run except `weight_decay`, which is raised
10x (1e-4 -> 1e-3). If this alone closes most of the gap between `train_loss`
and `val_loss` seen above, immediate overfitting was mainly an under-
regularized-optimizer problem, not an architecture-capacity problem --
useful to know before reaching for `AFTER_TUNING_CONFIG`, which changes both
the architecture AND every training hyperparameter at once and so can't
isolate which change did the work.

**Executed twice across real Colab sessions** -- both runs' full epoch logs
and a run-to-run variance comparison are in
`overfitting_adamw_comparison.xlsx` ("Regularized Run 2 Epoch Log" /
"Final Verdict"). The run shown below is this cell's most recent execution.


In [21]:
model_regularized, history_regularized = run_training(
    model_config=BEFORE_TUNING_CONFIG,
    train_config=TRAIN_REGULARIZED_CONFIG,
    windowed_dir=DEFAULT_WINDOWED_DIR,
    n_epochs=N_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
)
history_regularized  # compare val_f1_macro / val_loss trend against Section 10's run above

epoch   1/30 | train_loss 0.2847 | val_loss 0.2037 | val_f1_macro 0.7317 | val_hamming 0.0911
epoch   2/30 | train_loss 0.2044 | val_loss 0.1552 | val_f1_macro 0.7809 | val_hamming 0.0714
epoch   3/30 | train_loss 0.1823 | val_loss 0.1445 | val_f1_macro 0.8119 | val_hamming 0.0633
epoch   4/30 | train_loss 0.1673 | val_loss 0.2462 | val_f1_macro 0.7064 | val_hamming 0.1041
epoch   5/30 | train_loss 0.1563 | val_loss 0.1450 | val_f1_macro 0.8212 | val_hamming 0.0616
epoch   6/30 | train_loss 0.1458 | val_loss 0.1868 | val_f1_macro 0.7834 | val_hamming 0.0771
epoch   7/30 | train_loss 0.1373 | val_loss 0.1906 | val_f1_macro 0.7732 | val_hamming 0.0832
epoch   8/30 | train_loss 0.1304 | val_loss 0.1582 | val_f1_macro 0.7979 | val_hamming 0.0702
epoch   9/30 | train_loss 0.1241 | val_loss 0.1879 | val_f1_macro 0.7800 | val_hamming 0.0803
epoch  10/30 | train_loss 0.1200 | val_loss 0.1870 | val_f1_macro 0.7802 | val_hamming 0.0788
early stopping: no val_f1_macro improvement for 5 epochs


[{'epoch': 1,
  'train_loss': 0.2846695402557891,
  'val_f1_macro': 0.7316733558478011,
  'val_precision_macro': 0.7124617525473937,
  'val_recall_macro': 0.782880442694348,
  'val_hamming_loss': 0.09105076579687972,
  'val_jaccard_macro': np.float64(0.5892727592908769),
  'val_loss': 0.20365494474906004},
 {'epoch': 2,
  'train_loss': 0.2043725336218963,
  'val_f1_macro': 0.7809401927888219,
  'val_precision_macro': 0.7938088358866782,
  'val_recall_macro': 0.806713058847375,
  'val_hamming_loss': 0.07137809187279152,
  'val_jaccard_macro': np.float64(0.6631450333984096),
  'val_loss': 0.1552308656930226},
 {'epoch': 3,
  'train_loss': 0.18225126980757575,
  'val_f1_macro': 0.8119263856435621,
  'val_precision_macro': 0.8122095259117398,
  'val_recall_macro': 0.8364638752298734,
  'val_hamming_loss': 0.06329255552955011,
  'val_jaccard_macro': np.float64(0.7046656628098441),
  'val_loss': 0.14446210505784138},
 {'epoch': 4,
  'train_loss': 0.1673253430046877,
  'val_f1_macro': 0.70636

In [22]:
torch.save(model_regularized.state_dict(), "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/ma1dcnn_regularized.pt")
print("saved regularized model weights")

saved regularized model weights


### Optional: same regularized run, but with decoupled weight decay (AdamW)

Same `BEFORE_TUNING_CONFIG` + `TRAIN_REGULARIZED_CONFIG` as the cell above --
identical architecture, identical learning_rate/weight_decay/batch_size/threshold
-- the ONLY change is `run_training`'s new `optimizer_cls` argument:
`torch.optim.AdamW` instead of the default `torch.optim.Adam`.

`torch.optim.Adam`'s `weight_decay` is coupled L2 -- it's added into the
gradient before Adam's adaptive per-parameter scaling, so parameters with a
large historical gradient magnitude get regularized less than intended.
`torch.optim.AdamW` decouples the two, applying `weight_decay` directly to
each parameter (Loshchilov & Hutter, 2019, "Decoupled Weight Decay
Regularization," ICLR 2019). Since the regularized run above already showed
that `weight_decay=1e-3` helps, this cell tests whether the SAME weight_decay
value regularizes even more effectively once decoupled -- a clean,
single-variable follow-up.

**Executed twice across real Colab sessions, both WORSE than the regularized
run above** -- and not consistently worse in the same way (Run 1 degraded
slowly over 9 epochs, Run 2 degraded immediately and sharply over just 6).
Full comparison in `overfitting_adamw_comparison.xlsx`.


In [23]:
model_adamw, history_adamw = run_training(
    model_config=BEFORE_TUNING_CONFIG,
    train_config=TRAIN_REGULARIZED_CONFIG,
    windowed_dir=DEFAULT_WINDOWED_DIR,
    n_epochs=N_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
    optimizer_cls=torch.optim.AdamW,
)
history_adamw  # compare val_f1_macro / val_loss / gap trend against both runs above


epoch   1/30 | train_loss 0.2341 | val_loss 0.1820 | val_f1_macro 0.7902 | val_hamming 0.0710
epoch   2/30 | train_loss 0.1621 | val_loss 0.2339 | val_f1_macro 0.7737 | val_hamming 0.0794
epoch   3/30 | train_loss 0.1425 | val_loss 0.3019 | val_f1_macro 0.7381 | val_hamming 0.0987
epoch   4/30 | train_loss 0.1297 | val_loss 0.3874 | val_f1_macro 0.6943 | val_hamming 0.1209
epoch   5/30 | train_loss 0.1186 | val_loss 0.2815 | val_f1_macro 0.7747 | val_hamming 0.0806
epoch   6/30 | train_loss 0.1074 | val_loss 0.3579 | val_f1_macro 0.7548 | val_hamming 0.0899
early stopping: no val_f1_macro improvement for 5 epochs


[{'epoch': 1,
  'train_loss': 0.2340533694194626,
  'val_f1_macro': 0.7901781224982106,
  'val_precision_macro': 0.8015948549339633,
  'val_recall_macro': 0.7938195430731002,
  'val_hamming_loss': 0.07096936241629895,
  'val_jaccard_macro': np.float64(0.6674283491010939),
  'val_loss': 0.1820351522541365},
 {'epoch': 2,
  'train_loss': 0.1620744682939773,
  'val_f1_macro': 0.7736636324881626,
  'val_precision_macro': 0.7509706886206924,
  'val_recall_macro': 0.8164074933701301,
  'val_hamming_loss': 0.07942366863229762,
  'val_jaccard_macro': np.float64(0.6497289224764687),
  'val_loss': 0.23389615361728322},
 {'epoch': 3,
  'train_loss': 0.14246576900574465,
  'val_f1_macro': 0.7380857203625468,
  'val_precision_macro': 0.677765516247804,
  'val_recall_macro': 0.8199380427763956,
  'val_hamming_loss': 0.0986522203257277,
  'val_jaccard_macro': np.float64(0.5955158383275772),
  'val_loss': 0.301917456081827},
 {'epoch': 4,
  'train_loss': 0.1297179882051376,
  'val_f1_macro': 0.6943041

In [24]:
torch.save(model_adamw.state_dict(), "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/ma1dcnn_adamw.pt")
print("saved AdamW model weights")


saved AdamW model weights


### Optional: conv-stack dropout (same TRAIN_REGULARIZED_CONFIG, new architecture)

`BEFORE_TUNING_CONFIG_CONV_DROPOUT` (Section 3, synced from `ma1dcnn.ipynb`) +
`TRAIN_REGULARIZED_CONFIG` + plain `torch.optim.Adam` (the `optimizer_cls` default --
AdamW did not help, see the cell above). This is a single-variable test on the
ARCHITECTURE instead of the optimizer: dropout is now applied after each of the 3 conv
blocks (`conv_dropout=0.2`), not just once before the FC layer, so the feature-extraction
stage itself gets direct regularization pressure instead of relying entirely on
`weight_decay`'s blanket penalty on every weight.

**Run for real.** Early stopped at epoch 13 (the longest-running experiment yet).
Best checkpoint epoch 8 (`val_f1_macro=0.8177`) -- a close second to the Regularized
run's best, but with the tightest generalization gap of any run this session
(max +0.0288, under half the Regularized run's +0.0612). Full comparison in
`overfitting_adamw_comparison.xlsx`.


In [25]:
model_conv_dropout, history_conv_dropout = run_training(
    model_config=BEFORE_TUNING_CONFIG_CONV_DROPOUT,
    train_config=TRAIN_REGULARIZED_CONFIG,
    windowed_dir=DEFAULT_WINDOWED_DIR,
    n_epochs=N_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
)
history_conv_dropout  # compare val_f1_macro / val_loss / gap trend against every run above


epoch   1/30 | train_loss 0.2780 | val_loss 0.1741 | val_f1_macro 0.7701 | val_hamming 0.0750
epoch   2/30 | train_loss 0.2099 | val_loss 0.1518 | val_f1_macro 0.7936 | val_hamming 0.0662
epoch   3/30 | train_loss 0.1963 | val_loss 0.1813 | val_f1_macro 0.7670 | val_hamming 0.0783
epoch   4/30 | train_loss 0.1875 | val_loss 0.1587 | val_f1_macro 0.7772 | val_hamming 0.0726
epoch   5/30 | train_loss 0.1781 | val_loss 0.1407 | val_f1_macro 0.8065 | val_hamming 0.0626
epoch   6/30 | train_loss 0.1673 | val_loss 0.1835 | val_f1_macro 0.7935 | val_hamming 0.0764
epoch   7/30 | train_loss 0.1580 | val_loss 0.1711 | val_f1_macro 0.8041 | val_hamming 0.0731
epoch   8/30 | train_loss 0.1496 | val_loss 0.1442 | val_f1_macro 0.8177 | val_hamming 0.0647
epoch   9/30 | train_loss 0.1423 | val_loss 0.1711 | val_f1_macro 0.8072 | val_hamming 0.0729
epoch  10/30 | train_loss 0.1381 | val_loss 0.1624 | val_f1_macro 0.8167 | val_hamming 0.0687
epoch  11/30 | train_loss 0.1741 | val_loss 0.1830 | val_f1_

[{'epoch': 1,
  'train_loss': 0.2780487627360403,
  'val_f1_macro': 0.7700930882489032,
  'val_precision_macro': 0.7926404777047511,
  'val_recall_macro': 0.7879064253238777,
  'val_hamming_loss': 0.07496532078982972,
  'val_jaccard_macro': np.float64(0.647184961474975),
  'val_loss': 0.17406195937490251},
 {'epoch': 2,
  'train_loss': 0.20989838276104095,
  'val_f1_macro': 0.7936134670133439,
  'val_precision_macro': 0.838162660150158,
  'val_recall_macro': 0.788642482586995,
  'val_hamming_loss': 0.06623814770203851,
  'val_jaccard_macro': np.float64(0.6822364612934495),
  'val_loss': 0.15181131112730314},
 {'epoch': 3,
  'train_loss': 0.19634684228164392,
  'val_f1_macro': 0.7669945763826396,
  'val_precision_macro': 0.7624356784726964,
  'val_recall_macro': 0.7969904602186956,
  'val_hamming_loss': 0.07834932668101406,
  'val_jaccard_macro': np.float64(0.6392340332320564),
  'val_loss': 0.1813327353999863},
 {'epoch': 4,
  'train_loss': 0.18753334809237018,
  'val_f1_macro': 0.7772

In [26]:
torch.save(model_conv_dropout.state_dict(), "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/ma1dcnn_conv_dropout.pt")
print("saved conv-dropout model weights")


saved conv-dropout model weights


### Optional: BatchNorm1d after each conv block (same TRAIN_REGULARIZED_CONFIG, layered on conv-stack dropout)

`BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN` (Section 3, synced from `ma1dcnn.ipynb`) +
`TRAIN_REGULARIZED_CONFIG` + plain `torch.optim.Adam`. Both conv-stack dropout
(`conv_dropout=0.2`) and BatchNorm1d (`use_batchnorm=True`) are active at once here --
this is a single-variable test that adds ONE more regularization lever on top of the
conv-dropout cell above, rather than replacing it, so any change in the generalization
gap can be attributed to BatchNorm1d specifically.

**Not yet run for real.** `ma1dcnn.ipynb`'s own synthetic tests confirm the train/eval
running-statistics contract (batch stats + running-stat mutation in train mode, frozen
running stats in eval mode, matching the documented formula exactly) and that
`batch_size=1` is safe here specifically because `BatchNorm1d` normalizes a 3D
`(N, C, L)` input over `N*L` elements per channel, not `N` alone, and this project's
window length `L` is always > 1 -- so `TRAIN_REGULARIZED_CONFIG`'s `batch_size=128` is
nowhere near that edge case. Per this project's run-to-run-variance discipline, plan to
run this cell at least twice before trusting any single result.

In [ ]:
model_conv_dropout_bn, history_conv_dropout_bn = run_training(
    model_config=BEFORE_TUNING_CONFIG_CONV_DROPOUT_BN,
    train_config=TRAIN_REGULARIZED_CONFIG,
    windowed_dir=DEFAULT_WINDOWED_DIR,
    n_epochs=N_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
)
history_conv_dropout_bn  # compare val_f1_macro / val_loss / gap trend against every run above, especially the conv-dropout-only run


In [ ]:
torch.save(model_conv_dropout_bn.state_dict(), "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/ma1dcnn_conv_dropout_bn.pt")
print("saved conv-dropout+batchnorm model weights")


### Optional: Optuna trial 13 (knee-point pick from the completed 30-trial search)

`TRIAL_13_MODEL_CONFIG` + `TRIAL_13_TRAIN_CONFIG` (Section 3) -- the Pareto-optimal trial recommended in `training/optuna_search.ipynb`'s completed search, re-run here through the exact same `run_training()` already used by every cell above, for the same reason every one of those used it: a config found by Optuna is still only evaluated on `val_f1_macro` and the max generalization gap (design note 1 in `optuna_search.ipynb`'s own `objective()`), not the full metrics suite this notebook's `compute_metrics` produces every epoch (precision, recall, Hamming loss, Jaccard).

**Not yet run for real.** Running it will not reproduce the search's own `0.8375` / `-0.0188` exactly -- every config re-run in Section 10 above has shown real run-to-run variance on identical config and a fixed seed (non-bit-reproducible multi-threaded CPU BLAS), and this cell is no exception -- the point is to get the other four metrics for this configuration, then log the result into `overfitting_adamw_comparison.xlsx` alongside Trial 13's existing Optuna-only row.

In [ ]:
model_trial13, history_trial13 = run_training(
    model_config=TRIAL_13_MODEL_CONFIG,
    train_config=TRIAL_13_TRAIN_CONFIG,
    windowed_dir=DEFAULT_WINDOWED_DIR,
    n_epochs=N_EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    device=DEVICE,
)
history_trial13  # compare val_f1_macro / val_loss / gap trend against every run above, and against the Optuna-only numbers in overfitting_adamw_comparison.xlsx

In [ ]:
torch.save(model_trial13.state_dict(), "/content/drive/MyDrive/edge-ai-fault-diagnosis-aerospace/data/ma1dcnn_trial13.pt")
print("saved Optuna trial 13 model weights")

## What's next

Four of the five cells in Section 10 have now been run for real against `data/windowed/*.npz` -- several of them more than once across different Colab sessions -- see `overfitting_adamw_comparison.xlsx` for the complete run-by-run and run-to-run-variance breakdown. Summary of best checkpoints seen so far: Baseline (`BEFORE_TUNING_CONFIG` + `TRAIN_BEFORE_TUNING_CONFIG`) peaks immediately at epoch 1 (~0.79-0.80 `val_f1_macro` across its two runs) and overfits hard after; Regularized (`weight_decay` raised 10x) is the strongest and most consistent hand-run config (0.8085 / 0.8212 `val_f1_macro` across two runs, generalization gap 0.0612 / 0.0789); AdamW (same `weight_decay`, decoupled) is worse on both of its runs and inconsistent in HOW it's worse (slow 9-epoch decline vs. immediate 6-epoch collapse); Conv-Dropout (`conv_dropout=0.2` on all 3 conv blocks) is the best-CONTROLLED hand-run config -- a close-second best `val_f1_macro` (0.8177) with by far the tightest generalization gap (0.0288); BatchNorm1d layered on top raised the best hand-run `val_f1_macro` to 0.8274 but widened the gap on both of its runs instead of tightening it.

**`training/optuna_search.ipynb`'s 30-trial multi-objective search has since completed** and found a 5-trial Pareto front (trials 9, 1, 12, 13, 4) -- trials 12 and 13 both Pareto-dominate every hand-run config above outright (higher `val_f1_macro` AND a smaller, even negative, max gap than BatchNorm's own best). Trial 13 (`TRIAL_13_MODEL_CONFIG` / `TRIAL_13_TRAIN_CONFIG`, Section 3) is the recommended pick by knee-point analysis; its cell is just above this one.

Remaining work, in order:

1. **Run the Optuna trial 13 cell above for real.** This is the immediate next step -- it fills in `val_precision_macro`, `val_recall_macro`, `val_hamming_loss`, and `val_jaccard_macro` for the search's own recommended configuration, none of which `optuna_search.ipynb`'s `objective()` tracked. Log the result into `overfitting_adamw_comparison.xlsx` next to the existing Optuna-only row for this trial.
2. **Run Section 10 again with `AFTER_TUNING_CONFIG` / `TRAIN_AFTER_TUNING_CONFIG`** and compare against every result above -- this combination changes the architecture AND every training hyperparameter at once, so it can't isolate which change does the work the way the single-variable cells (and trial 13) can.
3. **Per `pipeline_sequence_and_dependencies.docx` Section 7**: `compression/` (quantization, pruning) and `deployment/` (Jetson Nano benchmarking) are the stages after this -- both reuse `build_dataloaders`, `train_one_epoch`, and `evaluate` directly rather than duplicating them.

Only once a tuned, non-overfitting configuration is confirmed -- trial 13 is the current candidate -- should any of this be ported into `src/edge_fault_dx/training/train.py`, following the same "prove it standalone first" discipline every notebook in this project has been built under.